# Moon & Mars process discovery with NASA PDS

**Goal:** use the NASA PDS Search API to rank planetary processes that can later become broad filters for Earth data.

This notebook intentionally does **not** search Earth yet. It downloads a controlled sample of PDS metadata, measures how much supporting material exists for each candidate process, shows representative observations, and exports a process shortlist.

The method is deliberately transparent: planetary processes are scientific interpretations, so the notebook starts with broad candidate families and uses PDS evidence to rank them. It does not pretend that an unsupervised image cluster automatically proves a geological process.

## What the notebook produces

- `pds_process_ranking.csv` — ranked Moon/Mars processes.
- `pds_process_evidence.csv` — representative PDS records supporting each process.
- `pds_process_shortlist.json` — compact machine-readable output for the later Earth-filtering notebook.

A high score means: the PDS catalog contains useful supporting observations, the process has measurable Earth signals, and it is useful for the analog-site challenge. It does **not** mean Earth perfectly reproduces Moon or Mars conditions.

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_columns', 30)

BASE_URL = 'https://pds.nasa.gov/api/search/1'
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
SAMPLE_PER_QUERY = 20
REQUEST_TIMEOUT = 90
PAUSE_BETWEEN_REQUESTS = 0.25

print('Output directory:', OUTPUT_DIR.resolve())
print('GPU is not required for this metadata-discovery stage.')

## Candidate processes

We begin with broad processes that are both important on the Moon/Mars and measurable using global Earth datasets. `earth_signals` describes what the later Earth notebook should search for. The two expert scores use a simple 1–5 scale and remain visible so the ranking is auditable.

In [ ]:
PROCESS_CANDIDATES = [
    {
        'planet': 'Mars', 'process': 'Aeolian dunes and dust transport',
        'search_terms': ['dune', 'aeolian', 'dust'],
        'earth_signals': ['very low rainfall', 'low vegetation', 'sand or dust', 'strong or persistent wind'],
        'earth_filterability': 5, 'challenge_value': 5,
        'why': 'Useful for rover mobility, dust exposure, landscape change, and desert analogs.'
    },
    {
        'planet': 'Mars', 'process': 'Fluvial erosion and sedimentation',
        'search_terms': ['channel', 'delta', 'fluvial'],
        'earth_signals': ['channels', 'deltas or fans', 'sedimentary deposits', 'dry or seasonal water flow'],
        'earth_filterability': 5, 'challenge_value': 5,
        'why': 'Connects ancient Martian water environments to terrestrial rivers, fans, deltas, and dry lake basins.'
    },
    {
        'planet': 'Mars', 'process': 'Water-driven mineral alteration',
        'search_terms': ['hydrated mineral', 'clay', 'sulfate'],
        'earth_signals': ['clay minerals', 'sulfates', 'salts', 'hydrothermal or evaporite geology'],
        'earth_filterability': 4, 'challenge_value': 5,
        'why': 'Strong evidence of past water and useful for geology, habitability, and biosignature studies.'
    },
    {
        'planet': 'Mars', 'process': 'Glacial and periglacial ice dynamics',
        'search_terms': ['glacial', 'permafrost', 'ground ice'],
        'earth_signals': ['permafrost', 'glaciers', 'freeze-thaw terrain', 'extreme cold and aridity'],
        'earth_filterability': 5, 'challenge_value': 5,
        'why': 'Relevant to water resources, buried ice, terrain stability, and cold-region operations.'
    },
    {
        'planet': 'Mars', 'process': 'Volcanism and lava emplacement',
        'search_terms': ['volcanic', 'lava flow', 'basalt'],
        'earth_signals': ['basaltic rock', 'lava fields', 'volcanic cones', 'young or exposed volcanic terrain'],
        'earth_filterability': 5, 'challenge_value': 4,
        'why': 'Provides geological and operational analogs in exposed basaltic landscapes.'
    },
    {
        'planet': 'Mars', 'process': 'Impact cratering and ejecta modification',
        'search_terms': ['impact crater', 'ejecta', 'impact'],
        'earth_signals': ['confirmed impact structures', 'ejecta or breccia', 'crater topography', 'fractured rock'],
        'earth_filterability': 3, 'challenge_value': 4,
        'why': 'Useful for terrain evolution, exposed subsurface materials, and mobility studies.'
    },
    {
        'planet': 'Moon', 'process': 'Impact cratering and regolith formation',
        'search_terms': ['impact crater', 'ejecta', 'regolith'],
        'earth_signals': ['impact structures', 'breccia or shattered rock', 'loose sediment', 'blocky crater terrain'],
        'earth_filterability': 3, 'challenge_value': 5,
        'why': 'The dominant lunar surface process and highly relevant to landing, excavation, and mobility.'
    },
    {
        'planet': 'Moon', 'process': 'Volcanism, pits, and lava tubes',
        'search_terms': ['lava tube', 'lunar pit', 'volcanic'],
        'earth_signals': ['basaltic lava fields', 'lava caves', 'collapse pits', 'tube-like subsurface voids'],
        'earth_filterability': 4, 'challenge_value': 5,
        'why': 'Relevant to volcanic history and possible protected subsurface base environments.'
    },
    {
        'planet': 'Moon', 'process': 'Polar cold trapping and volatile retention',
        'search_terms': ['permanently shadowed', 'polar ice', 'volatile'],
        'earth_signals': ['persistent shadow', 'extreme cold', 'surface or subsurface ice', 'low solar exposure'],
        'earth_filterability': 4, 'challenge_value': 5,
        'why': 'Directly relevant to polar resources, thermal extremes, illumination, and base planning.'
    },
    {
        'planet': 'Moon', 'process': 'Mass wasting on crater slopes',
        'search_terms': ['mass wasting', 'landslide', 'slope'],
        'earth_signals': ['steep bare slopes', 'rockfall', 'landslides', 'loose dry material'],
        'earth_filterability': 5, 'challenge_value': 4,
        'why': 'Useful for slope safety, rover navigation, excavation, and terrain-change studies.'
    }
]

candidate_df = pd.DataFrame(PROCESS_CANDIDATES)
candidate_df[['planet', 'process', 'search_terms', 'earth_signals', 'why']]

## PDS API client

The API is a large catalog. We request only a small sample for each search term and retain the total hit count reported by PDS. Results are cached locally so rerunning later cells does not repeat every network request.

In [ ]:
TARGET_IDS = {
    'Mars': 'urn:nasa:pds:context:target:planet.mars',
    'Moon': 'urn:nasa:pds:context:target:satellite.earth.moon',
}
CACHE_PATH = OUTPUT_DIR / 'pds_api_cache.json'

session = requests.Session()
session.headers.update({
    'Accept': 'application/json',
    'User-Agent': 'NASA-Space-Apps-Process-Discovery/1.0'
})

if CACHE_PATH.exists():
    try:
        api_cache = json.loads(CACHE_PATH.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, OSError):
        api_cache = {}
else:
    api_cache = {}

def pds_search(planet, keyword, limit=SAMPLE_PER_QUERY, retries=4):
    cache_key = json.dumps({'planet': planet, 'keyword': keyword, 'limit': limit}, sort_keys=True)
    if cache_key in api_cache:
        return api_cache[cache_key]

    params = {
        'q': f'(ref_lid_target eq \"{TARGET_IDS[planet]}\")',
        'keywords': keyword,
        'limit': limit,
    }
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(f'{BASE_URL}/products', params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            payload = response.json()
            api_cache[cache_key] = payload
            CACHE_PATH.write_text(json.dumps(api_cache), encoding='utf-8')
            time.sleep(PAUSE_BETWEEN_REQUESTS)
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'PDS request failed for {planet} / {keyword}: {last_error}')

def as_text(value):
    if value is None:
        return ''
    if isinstance(value, list):
        return ' | '.join(as_text(item) for item in value)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def normalize_record(record):
    metadata = record.get('metadata') or {}
    properties = record.get('properties') or {}
    descriptions = []
    if record.get('description'):
        descriptions.append(as_text(record['description']))
    for key, value in properties.items():
        if 'description' in key.lower() or 'abstract' in key.lower():
            descriptions.append(as_text(value))
    return {
        'product_id': as_text(record.get('id')),
        'product_type': as_text(record.get('type')),
        'title': as_text(record.get('title')),
        'description': ' | '.join(text for text in descriptions if text),
        'label_url': as_text(metadata.get('label_url')),
        'start_date_time': as_text(record.get('start_date_time')),
        'stop_date_time': as_text(record.get('stop_date_time')),
    }

print(f'Loaded {len(api_cache)} cached API responses.')

## Collect a reasonable sample

Default workload: 10 processes × 3 terms × 20 records, with a maximum of about 600 sampled records before duplicates. Only metadata is downloaded, so this is appropriate for Kaggle and avoids unexpectedly downloading multi-gigabyte scientific products.

In [ ]:
query_rows = []
evidence_rows = []

for candidate in PROCESS_CANDIDATES:
    print(f"Querying {candidate['planet']}: {candidate['process']}")
    for term in candidate['search_terms']:
        try:
            payload = pds_search(candidate['planet'], term)
            summary = payload.get('summary') or {}
            records = payload.get('data') or []
            hits = int(summary.get('hits') or 0)
            query_rows.append({
                'planet': candidate['planet'],
                'process': candidate['process'],
                'search_term': term,
                'catalog_hits': hits,
                'sample_returned': len(records),
                'error': '',
            })
            for record in records:
                normalized = normalize_record(record)
                normalized.update({
                    'planet': candidate['planet'],
                    'process': candidate['process'],
                    'search_term': term,
                })
                evidence_rows.append(normalized)
        except RuntimeError as exc:
            query_rows.append({
                'planet': candidate['planet'],
                'process': candidate['process'],
                'search_term': term,
                'catalog_hits': 0,
                'sample_returned': 0,
                'error': str(exc),
            })
            print('  Warning:', exc)

query_df = pd.DataFrame(query_rows)
evidence_df = pd.DataFrame(evidence_rows)
if not evidence_df.empty:
    evidence_df = evidence_df.drop_duplicates(['process', 'product_id']).reset_index(drop=True)

print(f'Finished {len(query_df)} API queries and retained {len(evidence_df)} unique evidence rows.')
display(query_df)

errors = query_df[query_df['error'] != '']
if not errors.empty:
    display(Markdown(f'**Warning:** {len(errors)} queries failed. Rankings still run, but inspect the errors above before trusting the affected processes.'))

## Rank the processes

The catalog evidence score uses the logarithm of hit counts so one extremely common keyword cannot dominate everything. The final score weights: **40% PDS evidence, 35% Earth filterability, 25% challenge value**. Search hit count measures availability, not scientific truth.

In [ ]:
if query_df.empty:
    raise RuntimeError('No PDS query results are available. Check Kaggle Internet access and rerun the collection cell.')

availability = (
    query_df.groupby(['planet', 'process'], as_index=False)
    .agg(
        catalog_hits=('catalog_hits', 'sum'),
        successful_terms=('catalog_hits', lambda values: int((values > 0).sum())),
        sampled_records=('sample_returned', 'sum'),
        failed_queries=('error', lambda values: int((values != '').sum())),
    )
)

ranking_df = candidate_df.merge(availability, on=['planet', 'process'], how='left')
for column in ['catalog_hits', 'successful_terms', 'sampled_records', 'failed_queries']:
    ranking_df[column] = ranking_df[column].fillna(0).astype(int)

log_hits = np.log1p(ranking_df['catalog_hits'].clip(lower=0))
if log_hits.max() > log_hits.min():
    ranking_df['pds_evidence_score'] = 100 * (log_hits - log_hits.min()) / (log_hits.max() - log_hits.min())
elif log_hits.max() > 0:
    ranking_df['pds_evidence_score'] = 100.0
else:
    ranking_df['pds_evidence_score'] = 0.0

ranking_df['earth_filter_score'] = ranking_df['earth_filterability'] * 20.0
ranking_df['challenge_score'] = ranking_df['challenge_value'] * 20.0
ranking_df['final_score'] = (
    0.40 * ranking_df['pds_evidence_score']
    + 0.35 * ranking_df['earth_filter_score']
    + 0.25 * ranking_df['challenge_score']
)
ranking_df.loc[ranking_df['successful_terms'] == 0, 'final_score'] *= 0.60
ranking_df['final_score'] = ranking_df['final_score'].round(1)
ranking_df['pds_evidence_score'] = ranking_df['pds_evidence_score'].round(1)
ranking_df = ranking_df.sort_values(['final_score', 'catalog_hits'], ascending=False).reset_index(drop=True)
ranking_df.insert(0, 'rank', np.arange(1, len(ranking_df) + 1))

display(ranking_df[[
    'rank', 'planet', 'process', 'final_score', 'catalog_hits',
    'successful_terms', 'sampled_records', 'earth_signals', 'why'
]])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_df = ranking_df.sort_values('final_score')
colors = plot_df['planet'].map({'Mars': '#c65d3b', 'Moon': '#8c939d'})
ax.barh(plot_df['process'], plot_df['final_score'], color=colors)
ax.set_xlabel('Priority score (0–100)')
ax.set_title('Processes to carry into the Earth-filtering stage')
ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.2)
for index, value in enumerate(plot_df['final_score']):
    ax.text(value + 0.8, index, f'{value:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Inspect the evidence

A process should not be kept only because its keyword is common. Review the representative titles and descriptions below. Product metadata can contain false positives, mission documents, or broad collection descriptions.

In [ ]:
def show_process_evidence(process, rows=8):
    if evidence_df.empty:
        print('No evidence records were returned.')
        return
    subset = evidence_df[evidence_df['process'] == process].copy()
    subset['text_length'] = subset['title'].str.len() + subset['description'].str.len()
    subset = subset.sort_values('text_length', ascending=False).head(rows)
    display(subset[[
        'planet', 'search_term', 'product_type', 'title', 'description',
        'product_id', 'label_url'
    ]])

for process in ranking_df.head(5)['process']:
    display(Markdown(f'### {process}'))
    show_process_evidence(process, rows=5)

## Final shortlist and exports

The shortlist keeps the top processes while ensuring that both planets are represented. It also includes the exact Earth signals needed by the next notebook.

In [ ]:
top_overall = ranking_df.head(6)
top_per_planet = ranking_df.groupby('planet', group_keys=False).head(2)
shortlist_df = (
    pd.concat([top_overall, top_per_planet], ignore_index=True)
    .drop_duplicates(['planet', 'process'])
    .sort_values(['final_score', 'catalog_hits'], ascending=False)
    .reset_index(drop=True)
)

ranking_path = OUTPUT_DIR / 'pds_process_ranking.csv'
evidence_path = OUTPUT_DIR / 'pds_process_evidence.csv'
shortlist_path = OUTPUT_DIR / 'pds_process_shortlist.json'

ranking_export = ranking_df.copy()
for column in ['search_terms', 'earth_signals']:
    ranking_export[column] = ranking_export[column].apply(lambda value: json.dumps(value, ensure_ascii=False))
ranking_export.to_csv(ranking_path, index=False)
evidence_df.to_csv(evidence_path, index=False)

shortlist_records = []
for row in shortlist_df.to_dict(orient='records'):
    shortlist_records.append({
        'planet': row['planet'],
        'process': row['process'],
        'priority_score': float(row['final_score']),
        'pds_catalog_hits': int(row['catalog_hits']),
        'search_terms': row['search_terms'],
        'earth_signals': row['earth_signals'],
        'why_it_matters': row['why'],
        'important_warning': 'PDS keyword hits are discovery evidence, not proof that every returned product measures this process.'
    })
shortlist_path.write_text(json.dumps(shortlist_records, indent=2, ensure_ascii=False), encoding='utf-8')

display(shortlist_df[['planet', 'process', 'final_score', 'earth_signals', 'why']])
print('Saved:', ranking_path)
print('Saved:', evidence_path)
print('Saved:', shortlist_path)

## How to use the result

The JSON shortlist is the handoff to the Earth stage. For example, if the shortlist keeps **aeolian dunes and dust transport**, the Earth search should combine low rainfall, low vegetation, exposed sediment, and wind. If it keeps **glacial and periglacial ice dynamics**, the Earth search should combine temperature, permafrost, snow/ice, aridity, and terrain.

Before publishing a scientific claim, manually inspect the evidence, replace broad catalog records with suitable instrument products, and cite the PDS product identifiers. The ranking is a defensible screening tool, not final geological proof.